In [2]:
#importing all the necessary libaries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

In [3]:
def scrape_flipkart_fashion(base_url, max_records=1200):

    # Import again inside function to ensure availability
    from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
    
    products = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept-Encoding': 'gzip, deflate, br',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }
    
    page = 1
    consecutive_failures = 0
    max_failures = 3
    
    print(f"Starting to scrape: {base_url}")
    print(f"Target: {max_records} records")
    print("=" * 70)
    
    while len(products) < max_records:
        try:
            # Parse the URL and add page parameter
            parsed_url = urlparse(base_url)
            query_params = parse_qs(parsed_url.query)
            
            # Add or update page parameter
            query_params['page'] = [str(page)]
            
            # Reconstruct URL with page parameter
            new_query = urlencode(query_params, doseq=True)
            page_url = urlunparse((
                parsed_url.scheme,
                parsed_url.netloc,
                parsed_url.path,
                parsed_url.params,
                new_query,
                parsed_url.fragment
            ))
            
            print(f"\n Scraping Page {page}...")
            print(f"Current count: {len(products)}/{max_records}")
            
            # Add random delay to avoid blocking
            time.sleep(random.uniform(2, 4))
            
            # Make request
            response = requests.get(page_url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"  Failed to fetch page {page}. Status: {response.status_code}")
                consecutive_failures += 1
                if consecutive_failures >= max_failures:
                    print(f" Too many failures. Stopping.")
                    break
                continue
            
            # Reset failure counter on success
            consecutive_failures = 0
            
            # Parse HTML
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find product containers using the correct parent element
            # Products are contained in divs that have the product name links
            product_links = soup.find_all('a', {'class': 'atJtCj'})
            
            if not product_links:
                # Try alternative class name
                product_links = soup.find_all('a', {'class': 'atJtCj Qum9aC'})
            
            # Get parent containers for each product
            product_cards = []
            for link in product_links:
                # Get the parent container that has all product info
                parent = link.find_parent('div', class_=lambda x: x and ('tUxRFH' in str(x) or 'CGtC98' in str(x) or 'slAVV4' in str(x)))
                if not parent:
                    parent = link.find_parent('div')
                if parent and parent not in product_cards:
                    product_cards.append(parent)
            
            if not product_cards or len(product_cards) == 0:
                print(f"  No products found on page {page}. Might have reached the end.")
                consecutive_failures += 1
                if consecutive_failures >= max_failures:
                    print("Stopping - no more products available.")
                    break
                page += 1
                continue
            
            print(f"   Found {len(product_cards)} product cards on this page")
            
            # Extract data from each product card
            page_products = 0
            for card in product_cards:
                if len(products) >= max_records:
                    break
                
                try:
                    product = {}
                    
                    # Product Name - using exact class names you provided
                    name_tag = card.find('a', {'class': 'atJtCj Qum9aC'})
                    if not name_tag:
                        name_tag = card.find('a', {'class': 'atJtCj'})
                    
                    product['name'] = name_tag.text.strip() if name_tag else 'N/A'
                    
                    # Skip if no name found
                    if product['name'] == 'N/A' or not product['name']:
                        continue
                    
                    # Company/Brand
                    company_tag = card.find('div', {'class': 'Fo1I0b'})
                    product['company'] = company_tag.text.strip() if company_tag else 'N/A'
                    
                    # Current Price
                    price_tag = card.find('div', {'class': 'hZ3P6w'})
                    product['price'] = price_tag.text.strip() if price_tag else 'N/A'
                    
                    # Original Price
                    original_price_tag = card.find('div', {'class': 'kRYCnD'})
                    product['original_price'] = original_price_tag.text.strip() if original_price_tag else 'N/A'
                    
                    # Discount
                    discount_tag = card.find('div', {'class': 'HQe8jr'})
                    product['discount'] = discount_tag.text.strip() if discount_tag else 'N/A'
                    
                    # Special Offers (Buy 2, Hot Deal, Only Few Left, etc.)
                    offers_tag = card.find('div', {'class': 'HZ0E6r Rm9_cy'})
                    if not offers_tag:
                        offers_tag = card.find('div', {'class': 'HZ0E6r'})
                    product['offers'] = offers_tag.text.strip() if offers_tag else 'N/A'
                    
                    # Size
                    size_tag = card.find('div', {'class': 'FD7OvP'})
                    product['size'] = size_tag.text.strip() if size_tag else 'N/A'
                    
                    # Product URL
                    if name_tag and 'href' in name_tag.attrs:
                        href = name_tag['href']
                        if href.startswith('/'):
                            product['url'] = 'https://www.flipkart.com' + href
                        else:
                            product['url'] = href
                    else:
                        product['url'] = 'N/A'
                    
                    # Page number
                    product['page'] = page
                    
                    # Add to products list
                    products.append(product)
                    page_products += 1
                    
                except Exception as e:
                    # Skip problematic products
                    continue
            
            print(f"✓ Extracted {page_products} products from page {page}")
            print(f"   Total so far: {len(products)}/{max_records}")
            
            # Move to next page
            page += 1
            
        except Exception as e:
            print(f" Error on page {page}: {e}")
            consecutive_failures += 1
            if consecutive_failures >= max_failures:
                print("Too many consecutive errors. Stopping.")
                break
            page += 1
            continue
    
    return products

print(" Scraping function defined successfully!")

✓ Scraping function defined successfully!


In [4]:
flipkart_url = "https://www.flipkart.com/clothing-and-accessories/topwear/pr?sid=clo,ash&p[]=facets.ideal_for%255B%255D%3DMen&p[]=facets.ideal_for%255B%255D%3Dmen&otracker=categorytree&fm=neo%2Fmerchandising&iid=M_cf6958ea-8f21-444b-a07c-1321c3aabf1a_1_X1NCR146KC29_MC.RLI1MOY42WPG&cid=RLI1MOY42WPG"

print("🚀 Starting Flipkart Fashion scraper...")
print("Category: Men's Topwear")
print("Note: This may take 10-15 minutes to extract 1200 records")
print("=" * 70)

# Run the scraper
scraped_data = scrape_flipkart_fashion(flipkart_url, max_records=1200)

# Convert to DataFrame
df = pd.DataFrame(scraped_data)

# Display summary
print("\n" + "=" * 70)
print(f"✓ SCRAPING COMPLETED!")
print(f"Total records extracted: {len(df)}")
print("=" * 70)

🚀 Starting Flipkart Fashion scraper...
Category: Men's Topwear
Note: This may take 10-15 minutes to extract 1200 records
Starting to scrape: https://www.flipkart.com/clothing-and-accessories/topwear/pr?sid=clo,ash&p[]=facets.ideal_for%255B%255D%3DMen&p[]=facets.ideal_for%255B%255D%3Dmen&otracker=categorytree&fm=neo%2Fmerchandising&iid=M_cf6958ea-8f21-444b-a07c-1321c3aabf1a_1_X1NCR146KC29_MC.RLI1MOY42WPG&cid=RLI1MOY42WPG
Target: 1200 records

📄 Scraping Page 1...
Current count: 0/1200
   Found 40 product cards on this page
✓ Extracted 40 products from page 1
   Total so far: 40/1200

📄 Scraping Page 2...
Current count: 40/1200
   Found 40 product cards on this page
✓ Extracted 40 products from page 2
   Total so far: 80/1200

📄 Scraping Page 3...
Current count: 80/1200
   Found 40 product cards on this page
✓ Extracted 40 products from page 3
   Total so far: 120/1200

📄 Scraping Page 4...
Current count: 120/1200
   Found 40 product cards on this page
✓ Extracted 40 products from page 4

KeyboardInterrupt: 

In [9]:
# Run the scraper
scraped_data = scrape_flipkart_fashion(flipkart_url, max_records=1000)

Starting to scrape: https://www.flipkart.com/clothing-and-accessories/topwear/pr?sid=clo,ash&p[]=facets.ideal_for%255B%255D%3DMen&p[]=facets.ideal_for%255B%255D%3Dmen&otracker=categorytree&fm=neo%2Fmerchandising&iid=M_cf6958ea-8f21-444b-a07c-1321c3aabf1a_1_X1NCR146KC29_MC.RLI1MOY42WPG&cid=RLI1MOY42WPG
Target: 1000 records

📄 Scraping Page 1...
Current count: 0/1000
   Found 40 product cards on this page
✓ Extracted 40 products from page 1
   Total so far: 40/1000

📄 Scraping Page 2...
Current count: 40/1000
   Found 40 product cards on this page
✓ Extracted 40 products from page 2
   Total so far: 80/1000

📄 Scraping Page 3...
Current count: 80/1000
   Found 40 product cards on this page
✓ Extracted 40 products from page 3
   Total so far: 120/1000

📄 Scraping Page 4...
Current count: 120/1000
   Found 40 product cards on this page
✓ Extracted 40 products from page 4
   Total so far: 160/1000

📄 Scraping Page 5...
Current count: 160/1000
   Found 40 product cards on this page
✓ Extract

In [10]:
# Convert to DataFrame
df = pd.DataFrame(scraped_data)

In [11]:
# Save to CSV
csv_filename = 'flipkart_mens_topwear_1000_records.csv'
df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print(f"✓ CSV saved: {csv_filename}")

✓ CSV saved: flipkart_mens_topwear_1200_records.csv
